# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display general metadata information
metadata = dataset.metadata
print(f"Name: {metadata.name}")
print(f"Version: {metadata.version}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We print out the available record sets with their summary metadata, then list fields for a sample record set.

> All lookups are by `@id`.

In [ ]:
# List all record sets with their @id and name
print('Available record sets:')
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"@id: {rs.id} | name: {rs.name}")

# Pick the first record set for field introspection
if len(record_sets) > 0:
    example_record_set = record_sets[0]
    print(f"\nFields in record set '@id': {example_record_set.id} (")
    for field in example_record_set.fields:
        print(f" @id: {field.id} | name: {field.name} | dataType: {field.data_type}")
else:
    print("No record sets found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

We'll extract all record sets listed above by their `@id`. Key names are resolved by their `@id` values.

In [ ]:
# Extract data from each record set
all_record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record set '@id': {record_set_id} loaded. Columns: {list(df.columns)} (rows: {len(df)})")

# Select a main record set for further demonstration (use the first one if available)
if all_record_set_ids:
    main_record_set_id = all_record_set_ids[0]
    print(f"\nColumns of main record set '@id': {main_record_set_id}")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print('No record sets available for data extraction.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's select a numeric field from the selected record set for filtering, normalization, and grouping.

All operations reference the appropriate `@id` for fields.

_Feel free to change the field IDs below as appropriate for the actual dataset as referenced above._

In [ ]:
# Example numeric field and group field (update these based on overview above)
if all_record_set_ids:
    record_set_id = main_record_set_id
    df = dataframes[record_set_id]

    # Infer a numeric field -- as an example pick the first column with numeric dtype
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

    if numeric_field_id is not None:
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.3f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a categorical field
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and (df[col].dtype == 'object' or pd.api.types.is_categorical_dtype(df[col])):
                group_field_id = col
                break

        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print('No suitable group field found.')
    else:
        print('No numeric field found in selected record set.')
else:
    print('No record sets loaded.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset with matplotlib/seaborn. All axes and legends use the `@id` names of fields.

_Modify the axes as appropriate based on the IDs shown above._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if all_record_set_ids and numeric_field_id is not None:
    # Histogram of numeric field
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    # If we have a group field
    if group_field_id is not None:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrates how to load, explore, and visualize the FAIR² protein mass spectrometry dataset using `mlcroissant`, referencing all dataset schema elements by their `@id`. You can further adapt EDA and modeling steps to answer biological or bioinformatics questions of interest.